In [ ]:
# ============================================================
# CELL 1 — IMPORTS AND REPRODUCIBILITY
# ============================================================

from pathlib import Path
import sys
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import statsmodels.api as sm

from dowhy import CausalModel
import dowhy

from IPython.display import display

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python version:", platform.python_version())
print("DoWhy version:", dowhy.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

In [ ]:
# ============================================================
# CELL 2 — PROJECT PATHS
# ============================================================
# Folder containing the full-cohort notebooks
EXPERIMENT_DIR = Path.cwd().resolve()

# Main project directory
PROJECT_ROOT = EXPERIMENT_DIR.parents[1]




DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

RESULTS_DIR = EXPERIMENT_DIR / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
GRAPHS_DIR = RESULTS_DIR / "graphs"

for directory in [
    TABLES_DIR,
    FIGURES_DIR,
    GRAPHS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from utils.visualization import visualize_causal_dag
except ImportError:
    visualize_causal_dag = None
    print(
        "Custom DAG visualization function was not imported. "
        "NetworkX visualization will remain available."
    )


print("Project root:", PROJECT_ROOT)
print("Processed-data directory:", PROCESSED_DIR)
print("Results directory:", RESULTS_DIR)

In [ ]:
# ============================================================
# CELL 2 — LOAD FULL-COHORT DOMAIN DATA
# ============================================================

FULL_DOMAIN_FILE = (
    PROCESSED_DIR
    / "attica_domain_scores_full.csv"
)

if not FULL_DOMAIN_FILE.exists():
    raise FileNotFoundError(
        f"Full domain dataset not found:\n"
        f"{FULL_DOMAIN_FILE}"
    )

df_causal = pd.read_csv(
    FULL_DOMAIN_FILE
)

DOMAIN_COLUMNS = [
    "Demographics",
    "SES",
    "Behavioral",
    "Treatment",
    "Obesity",
    "Metabolic",
    "Biomarkers",
    "Conditions",
    "Outcomes",
]

missing_columns = [
    column
    for column in DOMAIN_COLUMNS
    if column not in df_causal.columns
]

if missing_columns:
    raise ValueError(
        "Missing domain columns: "
        f"{missing_columns}"
    )

df_causal = df_causal[
    DOMAIN_COLUMNS
].copy()

for column in DOMAIN_COLUMNS:
    df_causal[column] = pd.to_numeric(
        df_causal[column],
        errors="coerce",
    )

df_causal = (
    df_causal
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=DOMAIN_COLUMNS)
    .reset_index(drop=True)
)

print("=" * 90)
print("FULL-COHORT CAUSAL DATASET")
print("=" * 90)

print("Shape:", df_causal.shape)

print(
    "Outcome distribution:",
    df_causal["Outcomes"]
    .value_counts()
    .sort_index()
    .to_dict(),
)

print(
    "Mortality percentage:",
    round(
        df_causal["Outcomes"].mean()
        * 100,
        3,
    ),
)

In [ ]:
# ============================================================
# CREATE TREATMENT TERTILES
# ============================================================
df_causal["DietTertile"] = pd.qcut(
    df_causal["Treatment"],
    q=3,
    labels=[
        "Lowest",
        "Middle",
        "Highest",
    ],
    duplicates="drop",
)

tertile_definition = (
    df_causal
    .groupby("DietTertile", observed=False)
    .agg(
        Participants=("Treatment", "size"),
        Treatment_min=("Treatment", "min"),
        Treatment_median=("Treatment", "median"),
        Treatment_max=("Treatment", "max"),
    )
    .reset_index()
)

display(tertile_definition.round(4))

In [ ]:
# ============================================================
# OBSERVED MORTALITY BY TREATMENT TERTILE
# ============================================================

tertile_results = (
    df_causal
    .groupby("DietTertile", observed=False)
    .agg(
        Participants=("Outcomes", "count"),
        Deaths=("Outcomes", "sum"),
        Mortality_rate=("Outcomes", "mean"),
        Treatment_median=("Treatment", "median"),
    )
    .reset_index()
)

tertile_results["Mortality (%)"] = (
    tertile_results["Mortality_rate"] * 100
)

display(
    tertile_results[
        [
            "DietTertile",
            "Participants",
            "Treatment_median",
            "Deaths",
            "Mortality (%)",
        ]
    ].round(3)
)

In [ ]:
from scipy.stats import chi2_contingency
# ============================================================
# CHI-SQUARE TEST OF ASSOCIATION
# ============================================================

contingency_table = pd.crosstab(
    df_causal["DietTertile"],
    df_causal["Outcomes"],
)

chi2, p_value, degrees_freedom, expected = (
    chi2_contingency(contingency_table)
)

print("Observed contingency table:")
display(contingency_table)

print("\nExpected counts under independence:")
display(
    pd.DataFrame(
        expected,
        index=contingency_table.index,
        columns=contingency_table.columns,
    ).round(2)
)

print(f"\nChi-square: {chi2:.3f}")
print(f"Degrees of freedom: {degrees_freedom}")
print(f"P-value: {p_value:.6g}")

In [ ]:
# ============================================================
# LINEAR TREND TEST ACROSS ORDERED TERTILES
# ============================================================

import statsmodels.api as sm

tertile_score_map = {
    "Lowest": 1,
    "Middle": 2,
    "Highest": 3,
}

df_causal["TertileScore"] = (
    df_causal["DietTertile"]
    .astype(str)
    .map(tertile_score_map)
)

trend_X = sm.add_constant(
    df_causal[["TertileScore"]],
    has_constant="add",
)

trend_model = sm.GLM(
    df_causal["Outcomes"],
    trend_X,
    family=sm.families.Binomial(),
).fit()

trend_beta = trend_model.params["TertileScore"]
trend_or = np.exp(trend_beta)
trend_ci = np.exp(
    trend_model.conf_int().loc["TertileScore"]
)
trend_p = trend_model.pvalues["TertileScore"]

print("Linear trend across diet tertiles")
print(f"Beta: {trend_beta:.4f}")
print(f"OR per higher tertile: {trend_or:.4f}")
print(
    "95% CI:",
    f"{trend_ci.iloc[0]:.4f}–{trend_ci.iloc[1]:.4f}",
)
print(f"P-value: {trend_p:.6g}")

In [ ]:
# ============================================================
# PLOT OBSERVED 10-YEAR MORTALITY BY DIET TERTILE
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    tertile_results["DietTertile"].astype(str),
    tertile_results["Mortality (%)"],
)

ax.set_title(
    "Observed ten-year all-cause mortality by baseline "
    "Mediterranean-diet adherence"
)

ax.set_xlabel(
    "Mediterranean-diet adherence tertile"
)

ax.set_ylabel(
    "Ten-year all-cause mortality (%)"
)

ax.grid(
    axis="y",
    alpha=0.25,
)

for bar, percentage, deaths, participants in zip(
    bars,
    tertile_results["Mortality (%)"],
    tertile_results["Deaths"],
    tertile_results["Participants"],
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{percentage:.2f}%\n"
        f"({int(deaths)}/{int(participants)})",
        ha="center",
        va="bottom",
    )

test_text = (
    f"Overall χ²({degrees_freedom}) = {chi2:.2f}, "
    f"p = {p_value:.3g}\n"
    f"Trend OR = {trend_or:.2f}, p = {trend_p:.3g}"
)

ax.text(
    0.98,
    0.96,
    test_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 3 — DEFINE CAUSAL QUERY
# ============================================================

TREATMENT = "Treatment"
OUTCOME = "Outcomes"

outcome_values = set(
    df_causal[OUTCOME]
    .dropna()
    .unique()
)

if not outcome_values.issubset({0, 1}):
    raise ValueError(
        f"{OUTCOME} is not binary: "
        f"{sorted(outcome_values)}"
    )

CONTROL_VALUE = float(
    df_causal[TREATMENT].quantile(0.25)
)

TREATMENT_VALUE = float(
    df_causal[TREATMENT].quantile(0.75)
)

TREATMENT_CONTRAST = (
    TREATMENT_VALUE
    - CONTROL_VALUE
)

print("=" * 90)
print("CAUSAL QUERY")
print("=" * 90)

print(
    "Treatment interpretation: "
    "higher Mediterranean-diet adherence"
)

print(
    "Outcome: ten-year cardiovascular mortality"
)

print(
    f"\nLower adherence intervention, Q1: "
    f"{CONTROL_VALUE:.6f}"
)

print(
    f"Higher adherence intervention, Q3: "
    f"{TREATMENT_VALUE:.6f}"
)

print(
    f"Intervention contrast: "
    f"{TREATMENT_CONTRAST:.6f}"
)

In [ ]:
# ============================================================
# CELL 4 — LOAD FULL-COHORT HYBRID PC DAG
# ============================================================

PRIMARY_GRAPH_FILE = (
    GRAPHS_DIR
    / "pc_hybrid_edges_full_cohort.csv"
)

if not PRIMARY_GRAPH_FILE.exists():
    raise FileNotFoundError(
        f"Primary graph not found:\n"
        f"{PRIMARY_GRAPH_FILE}\n\n"
        "Run the final export cell in Notebook 03."
    )

primary_edges_df = pd.read_csv(
    PRIMARY_GRAPH_FILE
)

required_columns = {
    "Source",
    "Target",
}

if not required_columns.issubset(
    primary_edges_df.columns
):
    raise ValueError(
        "The edge file must contain "
        "Source and Target columns."
    )

PRIMARY_EDGES = list(
    primary_edges_df[
        ["Source", "Target"]
    ].itertuples(
        index=False,
        name=None,
    )
)

primary_causal_dag = nx.DiGraph()

primary_causal_dag.add_nodes_from(
    DOMAIN_COLUMNS
)

primary_causal_dag.add_edges_from(
    PRIMARY_EDGES
)

if not nx.is_directed_acyclic_graph(
    primary_causal_dag
):
    raise ValueError(
        "The primary hybrid PC graph "
        "contains a directed cycle."
    )

if not nx.has_path(
    primary_causal_dag,
    TREATMENT,
    OUTCOME,
):
    raise ValueError(
        "No directed Treatment-to-Outcomes "
        "path exists."
    )

print("=" * 90)
print("FULL-COHORT HYBRID PC DAG")
print("=" * 90)

print(
    "Parents of Treatment:",
    sorted(
        primary_causal_dag.predecessors(
            TREATMENT
        )
    ),
)

print(
    "Parents of Outcomes:",
    sorted(
        primary_causal_dag.predecessors(
            OUTCOME
        )
    ),
)

print(
    "Descendants of Treatment:",
    sorted(
        nx.descendants(
            primary_causal_dag,
            TREATMENT,
        )
    ),
)

## Final causal model

The total causal effect of the Treatment domain on the Outcomes domain is
estimated using the final expert-informed domain DAG.

The target estimand is:

> The average change in the probability of 10-year cardiovascular mortality
> when the Treatment score is shifted from its 25th percentile to its 75th
> percentile.

The final DAG distinguishes pre-treatment confounders from variables lying on
the causal pathway. In particular, Obesity, Metabolic, Biomarkers, and
Conditions are not included in the adjustment set because they are descendants
of Treatment or downstream health states. Adjusting for them would block part
of the total causal effect.

The primary backdoor adjustment set is:

- Demographics

In [ ]:
# ============================================================
# IDENTIFY THE FULL-COHORT CAUSAL ESTIMAND
# ============================================================

from dowhy import CausalModel

causal_model_full = CausalModel(
    data=df_causal,
    treatment=TREATMENT,
    outcome=OUTCOME,
    graph=primary_causal_dag,
)

identified_estimand_full = (
    causal_model_full.identify_effect(
        proceed_when_unidentifiable=False
    )
)

backdoor_variables_full = sorted(
    identified_estimand_full.get_backdoor_variables()
)

print("=" * 90)
print("FULL-COHORT IDENTIFIED ESTIMAND")
print("=" * 90)

print(identified_estimand_full)

print("\nBackdoor adjustment variables:")
print(backdoor_variables_full)

print(
    "\nExpected from the current graph: "
    "Demographics should be identified."
)

In [ ]:
# ============================================================
# CELL 6 — VALIDATE THE ADJUSTMENT SET
# ============================================================

ADJUSTMENT_SET = sorted(
    identified_estimand_full.get_backdoor_variables()
)

treatment_descendants = sorted(
    set(
        nx.descendants(
            primary_causal_dag,
            TREATMENT,
        )
    )
    - {OUTCOME}
)

invalid_adjustments = sorted(
    set(ADJUSTMENT_SET)
    .intersection(treatment_descendants)
)

print("=" * 90)
print("ADJUSTMENT-SET VALIDATION")
print("=" * 90)

print("Identified adjustment set:")
print(ADJUSTMENT_SET)

print("\nPost-treatment descendants:")
print(treatment_descendants)

print("\nPost-treatment variables selected for adjustment:")
print(invalid_adjustments)

if invalid_adjustments:
    raise ValueError(
        "The adjustment set contains post-treatment "
        f"variables: {invalid_adjustments}"
    )

if not ADJUSTMENT_SET:
    print(
        "No measured backdoor adjustment "
        "variables were identified."
    )
else:
    print(
        "Identified backdoor adjustment set:",
        ADJUSTMENT_SET,
    )

print(
    "\nValidation passed: no post-treatment "
    "descendants are included."
)

In [ ]:
# ============================================================
# CELL 7 — ESTIMATE ADJUSTED LOGISTIC MODEL
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# Variables included in the outcome model
MODEL_VARIABLES = [
    TREATMENT,
    *ADJUSTMENT_SET,
]

# Keep only complete observations required for estimation
analysis_df = (
    df_causal[
        MODEL_VARIABLES + [OUTCOME]
    ]
    .dropna()
    .reset_index(drop=True)
)

# Design matrix with intercept
X = sm.add_constant(
    analysis_df[MODEL_VARIABLES],
    has_constant="add",
)

# Binary mortality outcome
y = analysis_df[OUTCOME].astype(int)

# Adjusted logistic regression
logistic_model_full = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
).fit(
    cov_type="HC3"
)

print("=" * 90)
print("FULL-COHORT ADJUSTED LOGISTIC MODEL")
print("=" * 90)

print("Sample size:", len(analysis_df))
print("Mortality events:", int(y.sum()))
print("Adjustment set:", ADJUSTMENT_SET)
print("Model variables:", MODEL_VARIABLES)

print("\n")
print(logistic_model_full.summary())

# ============================================================
# TREATMENT COEFFICIENT RESULTS
# ============================================================

treatment_beta = float(
    logistic_model_full.params[TREATMENT]
)

treatment_se = float(
    logistic_model_full.bse[TREATMENT]
)

treatment_pvalue = float(
    logistic_model_full.pvalues[TREATMENT]
)

treatment_ci = (
    logistic_model_full.conf_int()
    .loc[TREATMENT]
)

treatment_ci_low = float(
    treatment_ci.iloc[0]
)

treatment_ci_high = float(
    treatment_ci.iloc[1]
)

# Odds ratio for a one-unit increase in Treatment
treatment_or_per_unit = float(
    np.exp(treatment_beta)
)

treatment_or_ci_low_per_unit = float(
    np.exp(treatment_ci_low)
)

treatment_or_ci_high_per_unit = float(
    np.exp(treatment_ci_high)
)

print("\n" + "=" * 90)
print("ADJUSTED TREATMENT COEFFICIENT")
print("=" * 90)

print(
    "Log-odds coefficient:",
    round(treatment_beta, 6),
)

print(
    "Robust standard error:",
    round(treatment_se, 6),
)

print(
    "P-value:",
    round(treatment_pvalue, 6),
)

print(
    "Adjusted OR for a full 0-to-1 increase in adherence:",
    round(treatment_or_per_unit, 6),
)

print(
    "95% CI:",
    (
        round(treatment_or_ci_low_per_unit, 6),
        round(treatment_or_ci_high_per_unit, 6),
    ),
)

if treatment_beta < 0:
    print(
        "\nDirection: higher Mediterranean-diet adherence "
        "is associated with lower adjusted mortality odds."
    )
elif treatment_beta > 0:
    print(
        "\nDirection: higher Mediterranean-diet adherence "
        "is associated with higher adjusted mortality odds."
    )
else:
    print(
        "\nDirection: no estimated treatment association."
    )

In [ ]:
# ============================================================
# CELL 8 — STANDARDIZED Q1-TO-Q3 EFFECT
# ============================================================

LOW_ADHERENCE_VALUE = float(
    analysis_df[TREATMENT].quantile(0.25)
)

HIGH_ADHERENCE_VALUE = float(
    analysis_df[TREATMENT].quantile(0.75)
)

TREATMENT_DIFFERENCE = (
    HIGH_ADHERENCE_VALUE
    - LOW_ADHERENCE_VALUE
)

# Counterfactual dataset under lower adherence
data_low = analysis_df.copy()

data_low[TREATMENT] = (
    LOW_ADHERENCE_VALUE
)

X_low = sm.add_constant(
    data_low[MODEL_VARIABLES],
    has_constant="add",
)

# Counterfactual dataset under higher adherence
data_high = analysis_df.copy()

data_high[TREATMENT] = (
    HIGH_ADHERENCE_VALUE
)

X_high = sm.add_constant(
    data_high[MODEL_VARIABLES],
    has_constant="add",
)

predicted_risk_low = (
    logistic_model_full.predict(X_low)
)

predicted_risk_high = (
    logistic_model_full.predict(X_high)
)

standardized_risk_low = float(
    predicted_risk_low.mean()
)

standardized_risk_high = float(
    predicted_risk_high.mean()
)

# Negative means lower mortality under higher adherence
risk_difference = (
    standardized_risk_high
    - standardized_risk_low
)

risk_ratio = (
    standardized_risk_high
    / standardized_risk_low
)

# Odds ratio for Q3 compared with Q1
treatment_beta = float(
    logistic_model_full.params[TREATMENT]
)

treatment_se = float(
    logistic_model_full.bse[TREATMENT]
)

treatment_pvalue = float(
    logistic_model_full.pvalues[TREATMENT]
)

log_or = (
    treatment_beta
    * TREATMENT_DIFFERENCE
)

log_or_low = (
    treatment_beta - 1.96 * treatment_se
) * TREATMENT_DIFFERENCE

log_or_high = (
    treatment_beta + 1.96 * treatment_se
) * TREATMENT_DIFFERENCE

odds_ratio = float(
    np.exp(log_or)
)

odds_ratio_ci_values = sorted([
    float(np.exp(log_or_low)),
    float(np.exp(log_or_high)),
])

odds_ratio_ci_low = (
    odds_ratio_ci_values[0]
)

odds_ratio_ci_high = (
    odds_ratio_ci_values[1]
)

print("=" * 90)
print("FULL-COHORT STANDARDIZED EFFECT")
print("=" * 90)

print(
    f"Participants: {len(analysis_df)}"
)

print(
    f"Mortality events: {int(y.sum())}"
)

print(
    f"\nLower adherence, Q1: "
    f"{LOW_ADHERENCE_VALUE:.6f}"
)

print(
    f"Higher adherence, Q3: "
    f"{HIGH_ADHERENCE_VALUE:.6f}"
)

print(
    f"Treatment contrast: "
    f"{TREATMENT_DIFFERENCE:.6f}"
)

print(
    f"\nStandardized mortality risk at Q1: "
    f"{standardized_risk_low:.4%}"
)

print(
    f"Standardized mortality risk at Q3: "
    f"{standardized_risk_high:.4%}"
)

print(
    f"\nRisk difference, Q3 − Q1: "
    f"{risk_difference:.6f}"
)

print(
    f"Risk ratio, Q3 / Q1: "
    f"{risk_ratio:.4f}"
)

print(
    f"\nAdjusted odds ratio, Q3 vs Q1: "
    f"{odds_ratio:.4f}"
)

print(
    f"95% CI: "
    f"[{odds_ratio_ci_low:.4f}, "
    f"{odds_ratio_ci_high:.4f}]"
)

print(
    f"Treatment coefficient p-value: "
    f"{treatment_pvalue:.6f}"
)

if risk_difference < 0:
    print(
        "\nInterpretation: higher Mediterranean-diet "
        "adherence is associated with lower estimated "
        "ten-year cardiovascular mortality."
    )
else:
    print(
        "\nInterpretation: higher Mediterranean-diet "
        "adherence is associated with higher estimated "
        "ten-year cardiovascular mortality."
    )

In [ ]:
# ============================================================
# CELL 9 — SAVE FULL-COHORT EFFECT RESULTS
# ============================================================

full_cohort_effect_result = pd.DataFrame([
    {
        "Model": (
            "Full-cohort hybrid PC DAG"
        ),
        "Estimator": (
            "Binomial GLM with standardization"
        ),
        "Sample_Size": len(analysis_df),
        "Mortality_Events": int(y.sum()),
        "Adjustment_Set": ", ".join(
            ADJUSTMENT_SET
        ),
        "Treatment_Direction": (
            "Higher values indicate better "
            "Mediterranean-diet adherence"
        ),
        "Q1_Lower_Adherence": (
            LOW_ADHERENCE_VALUE
        ),
        "Q3_Higher_Adherence": (
            HIGH_ADHERENCE_VALUE
        ),
        "Treatment_Contrast": (
            TREATMENT_DIFFERENCE
        ),
        "Risk_Q1": (
            standardized_risk_low
        ),
        "Risk_Q3": (
            standardized_risk_high
        ),
        "Risk_Difference_Q3_minus_Q1": (
            risk_difference
        ),
        "Risk_Ratio_Q3_over_Q1": (
            risk_ratio
        ),
        "Odds_Ratio_Q3_vs_Q1": (
            odds_ratio
        ),
        "OR_CI_Lower": (
            odds_ratio_ci_low
        ),
        "OR_CI_Upper": (
            odds_ratio_ci_high
        ),
        "Treatment_P_Value": (
            treatment_pvalue
        ),
    }
])

FULL_EFFECT_RESULT_FILE = (
    TABLES_DIR
    / "full_cohort_hybrid_pc_effect.csv"
)

full_cohort_effect_result.to_csv(
    FULL_EFFECT_RESULT_FILE,
    index=False,
)

display(full_cohort_effect_result)

print("\nSaved to:")
print(FULL_EFFECT_RESULT_FILE)

In [ ]:
# ============================================================
# BOOTSTRAP CONFIDENCE INTERVALS FOR STANDARDIZED EFFECTS
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

N_BOOTSTRAP = 1000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

bootstrap_results = []

n = len(analysis_df)

print("=" * 90)
print("BOOTSTRAP STANDARDIZED EFFECT ESTIMATION")
print("=" * 90)

print("Bootstrap replications:", N_BOOTSTRAP)
print("Participants:", n)
print("Treatment contrast:", TREATMENT_DIFFERENCE)

# ------------------------------------------------------------
# Bootstrap loop
# ------------------------------------------------------------

for bootstrap_index in range(N_BOOTSTRAP):

    # Sample participants with replacement
    sampled_indices = rng.integers(
        low=0,
        high=n,
        size=n,
    )

    bootstrap_df = (
        analysis_df
        .iloc[sampled_indices]
        .reset_index(drop=True)
    )

    try:
        # ----------------------------------------
        # Fit bootstrap-adjusted logistic model
        # ----------------------------------------

        X_bootstrap = sm.add_constant(
            bootstrap_df[MODEL_VARIABLES],
            has_constant="add",
        )

        y_bootstrap = (
            bootstrap_df[OUTCOME]
            .astype(int)
        )

        bootstrap_model = sm.GLM(
            y_bootstrap,
            X_bootstrap,
            family=sm.families.Binomial(),
        ).fit()

        # ----------------------------------------
        # Set Treatment to Q1 and Q3
        # ----------------------------------------

        bootstrap_q1 = bootstrap_df.copy()
        bootstrap_q3 = bootstrap_df.copy()

        bootstrap_q1[TREATMENT] = (
            LOW_ADHERENCE_VALUE
        )

        bootstrap_q3[TREATMENT] = (
            HIGH_ADHERENCE_VALUE
        )

        X_bootstrap_q1 = sm.add_constant(
            bootstrap_q1[MODEL_VARIABLES],
            has_constant="add",
        )

        X_bootstrap_q3 = sm.add_constant(
            bootstrap_q3[MODEL_VARIABLES],
            has_constant="add",
        )

        # ----------------------------------------
        # Standardized risks
        # ----------------------------------------

        bootstrap_risk_q1 = float(
            bootstrap_model
            .predict(X_bootstrap_q1)
            .mean()
        )

        bootstrap_risk_q3 = float(
            bootstrap_model
            .predict(X_bootstrap_q3)
            .mean()
        )

        bootstrap_risk_difference = (
            bootstrap_risk_q3
            - bootstrap_risk_q1
        )

        bootstrap_risk_ratio = (
            bootstrap_risk_q3
            / bootstrap_risk_q1
            if bootstrap_risk_q1 > 0
            else np.nan
        )

        treatment_beta_bootstrap = float(
            bootstrap_model.params[TREATMENT]
        )

        bootstrap_odds_ratio = float(
            np.exp(
                treatment_beta_bootstrap
                * TREATMENT_DIFFERENCE
            )
        )

        bootstrap_results.append({
            "Risk_Q1": bootstrap_risk_q1,
            "Risk_Q3": bootstrap_risk_q3,
            "Risk_Difference": (
                bootstrap_risk_difference
            ),
            "Risk_Ratio": bootstrap_risk_ratio,
            "Odds_Ratio": bootstrap_odds_ratio,
        })

    except Exception:
        # Skip failed bootstrap samples
        continue

# ------------------------------------------------------------
# Convert bootstrap results to dataframe
# ------------------------------------------------------------

bootstrap_results_df = pd.DataFrame(
    bootstrap_results
)

successful_bootstraps = len(
    bootstrap_results_df
)

print(
    "\nSuccessful bootstrap replications:",
    successful_bootstraps,
)

if successful_bootstraps < 500:
    raise RuntimeError(
        "Too many bootstrap models failed. "
        "Inspect outcome variation and model stability."
    )

# ------------------------------------------------------------
# Percentile confidence intervals
# ------------------------------------------------------------

def percentile_ci(series, alpha=0.05):
    lower = float(
        series.quantile(alpha / 2)
    )

    upper = float(
        series.quantile(1 - alpha / 2)
    )

    return lower, upper


risk_q1_ci_low, risk_q1_ci_high = (
    percentile_ci(
        bootstrap_results_df["Risk_Q1"]
    )
)

risk_q3_ci_low, risk_q3_ci_high = (
    percentile_ci(
        bootstrap_results_df["Risk_Q3"]
    )
)

risk_difference_ci_low, risk_difference_ci_high = (
    percentile_ci(
        bootstrap_results_df[
            "Risk_Difference"
        ]
    )
)

risk_ratio_ci_low, risk_ratio_ci_high = (
    percentile_ci(
        bootstrap_results_df[
            "Risk_Ratio"
        ]
    )
)

odds_ratio_boot_ci_low, odds_ratio_boot_ci_high = (
    percentile_ci(
        bootstrap_results_df[
            "Odds_Ratio"
        ]
    )
)

# ------------------------------------------------------------
# Display final bootstrap estimates
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("BOOTSTRAP STANDARDIZED EFFECT RESULTS")
print("=" * 90)

print(
    f"Mortality risk at Q1: "
    f"{standardized_risk_low:.4%}"
)

print(
    f"95% bootstrap CI: "
    f"[{risk_q1_ci_low:.4%}, "
    f"{risk_q1_ci_high:.4%}]"
)

print(
    f"\nMortality risk at Q3: "
    f"{standardized_risk_high:.4%}"
)

print(
    f"95% bootstrap CI: "
    f"[{risk_q3_ci_low:.4%}, "
    f"{risk_q3_ci_high:.4%}]"
)

print(
    f"\nRisk difference, Q3 − Q1: "
    f"{risk_difference:.6f}"
)

print(
    f"95% bootstrap CI: "
    f"[{risk_difference_ci_low:.6f}, "
    f"{risk_difference_ci_high:.6f}]"
)

print(
    f"\nRisk ratio, Q3 / Q1: "
    f"{risk_ratio:.4f}"
)

print(
    f"95% bootstrap CI: "
    f"[{risk_ratio_ci_low:.4f}, "
    f"{risk_ratio_ci_high:.4f}]"
)

print(
    f"\nAdjusted odds ratio, Q3 vs Q1: "
    f"{odds_ratio:.4f}"
)

print(
    f"95% bootstrap CI: "
    f"[{odds_ratio_boot_ci_low:.4f}, "
    f"{odds_ratio_boot_ci_high:.4f}]"
)

# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------

bootstrap_summary = pd.DataFrame([
    {
        "Measure": "Risk at Q1",
        "Estimate": standardized_risk_low,
        "CI_Lower": risk_q1_ci_low,
        "CI_Upper": risk_q1_ci_high,
    },
    {
        "Measure": "Risk at Q3",
        "Estimate": standardized_risk_high,
        "CI_Lower": risk_q3_ci_low,
        "CI_Upper": risk_q3_ci_high,
    },
    {
        "Measure": "Risk difference Q3-Q1",
        "Estimate": risk_difference,
        "CI_Lower": risk_difference_ci_low,
        "CI_Upper": risk_difference_ci_high,
    },
    {
        "Measure": "Risk ratio Q3/Q1",
        "Estimate": risk_ratio,
        "CI_Lower": risk_ratio_ci_low,
        "CI_Upper": risk_ratio_ci_high,
    },
    {
        "Measure": "Odds ratio Q3/Q1",
        "Estimate": odds_ratio,
        "CI_Lower": odds_ratio_boot_ci_low,
        "CI_Upper": odds_ratio_boot_ci_high,
    },
])

display(bootstrap_summary)

In [ ]:
print(
    df_causal.groupby("Outcomes")["Treatment"]
    .agg(["count", "mean", "median", "min", "max"])
)

In [ ]:
# ============================================================
# SAVE BOOTSTRAP RESULTS
# ============================================================

BOOTSTRAP_SUMMARY_FILE = (
    TABLES_DIR
    / "full_cohort_bootstrap_effect_summary.csv"
)

BOOTSTRAP_DISTRIBUTION_FILE = (
    TABLES_DIR
    / "full_cohort_bootstrap_distribution.csv"
)

bootstrap_summary.to_csv(
    BOOTSTRAP_SUMMARY_FILE,
    index=False,
)

bootstrap_results_df.to_csv(
    BOOTSTRAP_DISTRIBUTION_FILE,
    index=False,
)

print("Saved bootstrap summary to:")
print(BOOTSTRAP_SUMMARY_FILE)

print("\nSaved bootstrap distribution to:")
print(BOOTSTRAP_DISTRIBUTION_FILE)